## Run SUMMA model ##

Run the SUMMA base model, with the model runs managed by Snakemake


In [1]:

from pathlib import Path

import sys
sys.path.append('../scripts')
from ss_utils import resolve_paths

### Write snakemake file ###

This block of code writes the snakemake file

In [84]:
config_file = "/Users/dcasson/GitHub/snow_dist/snakemake/config/chena_above_two_rivers.yaml"

In [354]:
%%writefile ../rules/run_pysumma_base.smk
"""

Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user.

"""

from pathlib import Path
import sys
import pandas as pd
import pysumma as ps

sys.path.append('../scripts/')
import ss_utils

ruleorder: run_summa_in_gru_chunks > merge_restart_files_per_date

config = ss_utils.promote_keys(config)

attributes_nc = Path(config['model_settings_path'],config['attributes_nc'])
num_grus = ss_utils.calc_num_grus(attributes_nc)
gru_chunk_strings = ss_utils.generate_gru_start_and_count(num_grus, chunk_size=config['gru_chunk_size'])

config['sim_start_time'] = "2010-10-02 00:00"
config['sim_end_time'] = "2011-09-30 23:00"
# Generate list of dates at monthly intervals
sim_start_time = pd.to_datetime(config['sim_start_time'])
sim_end_time = pd.to_datetime(config['sim_end_time'])
DATES = pd.date_range(start=sim_start_time, end=sim_end_time, freq='MS')
DATE_STRINGS = [date.strftime('%Y%m%d%H') for date in DATES]
print(DATE_STRINGS)

rule run_summa_base_simulations:
    input:
        Path(config['summa_output_path'],f"{config['case_name']}_{config['run_suffix']}_timestep.nc"),
        merged_restart_file = expand(Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}.nc"), date_str=DATE_STRINGS)
        
rule run_summa_in_gru_chunks:
    input:
        file_manager = Path(config['file_manager'])
    output:
        summa_chunked_output = Path(config['summa_output_path'],f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc"),
    params:
        summa_exe = config['summa_exe'],
        gru_start = lambda wildcards: ss_utils.extract_gru_int({wildcards.gru_chunk}),
        gru_count = config['gru_chunk_size']
    log:
        log_file = Path(config['logging_path'],f"summa_{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}.log")
    run:
        sim = ps.Simulation(params.summa_exe,input.file_manager)
        sim_config = {'manager': {'simStartTime' : config['sim_start_time'],
                              'simEndTime'  : config['sim_end_time']}
        }
        sim.apply_config(sim_config)

        if params.gru_start + params.gru_count > num_grus:
            params.gru_count = num_grus - params.gru_start + 1

        sim.run(run_suffix=config['run_suffix'], startGRU=params.gru_start, countGRU=params.gru_count, write_config=False,write_manager=True,freq_restart='m')

        print(f"SUMMA simulation for GRUs {params.gru_start} to {params.gru_start + params.gru_count - 1} finished with {sim.status}.")

rule merge_restart_files_per_date:
    input:
        chunked_restart_files = dynamic([Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}_{gru_chunk_wildcard}.nc") for
        gru_chunk_wildcard in gru_chunk_strings])
    output:
        merged_restart_file = Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}.nc")
    run:
        ss_utils.merge_netcdf_output(input.chunked_restart_files, output.merged_restart_file)


rule merge_summa_output:
    input:
        chunked_model_run = expand(Path(config['summa_output_path'],f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc"),gru_chunk=gru_chunk_strings),
    output:
        merged_model_run = Path(config['summa_output_path'],f"{config['case_name']}_{config['run_suffix']}_timestep.nc")

Overwriting ../rules/run_pysumma_base.smk


In [357]:
#! snakemake --unlock -s ../rules/run_pysumma_base.smk --configfile {config_file}
! snakemake -s ../rules/run_pysumma_base.smk --cores 8 --configfile {config_file} --nolock --forceall


Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 8
Rules claiming more threads will be scaled down.
Job stats:
job                             count
----------------------------  -------
all                                 1
generate_restart_file             180
merge_restart_files_per_date       12
merge_summa_output_files            1
run_summa_in_gru_chunks            15
total                             209

Select jobs to execute...

[Thu Oct  3 14:03:46 2024]
rule run_summa_in_gru_chunks:
    input: /Users/dcasson/Data/yukon_esp/summa/settings/fileManager.txt
    output: /Users/dcasson/Data/yukon_esp/output/Chena_base_G28-30_timestep.nc
    log: /Users/dcasson/Data/yukon_esp/output/logs/summa_Chena_base_G28-30.log
    jobid: 21
    reason: Forced execution
    wildcards: gru_chunk=G28-30
    resources: tmpdir=/var/folders/w4/qfhdls_x5jj48jhgnhtfv1lm0000gn/T

[Thu Oct  3 14:03:46 2024]
rule run_summa_in_gru_chunks:
    input: /Users/dcasson/Data/yukon_esp/summa/setti

In [356]:
%%writefile ../rules/run_pysumma_base.smk
"""
Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user.
"""

from pathlib import Path
import sys
import pandas as pd
import pysumma as ps

sys.path.append('../scripts/')
import ss_utils

# Promote config keys
config = ss_utils.promote_keys(config)

# Calculate the number of GRUs and generate chunk strings
attributes_nc = Path(config['model_settings_path'], config['attributes_nc'])
num_grus = ss_utils.calc_num_grus(attributes_nc)
GRU_CHUNKS = ss_utils.generate_gru_start_and_count(
    num_grus, chunk_size=config['gru_chunk_size']
)

# Simulation start and end times
config['sim_start_time'] = "2010-10-01 00:00"
config['sim_end_time'] = "2011-09-30 23:00"

# Generate list of dates at monthly intervals
sim_start_time = pd.to_datetime(config['sim_start_time'])
sim_end_time = pd.to_datetime(config['sim_end_time'])
DATES = pd.date_range(start=sim_start_time, end=sim_end_time, freq='MS')
DATE_STRINGS = [date.strftime('%Y%m%d%H') for date in DATES]

rule all:
    input:
        # Merged restart files for all dates
        #expand(
        #    Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}.nc"),
        #    date_str=DATE_STRINGS
        #),
        # Merged timestep output file
        Path(config['summa_output_path'], f"{config['case_name']}_{config['run_suffix']}_timestep.nc")

# Rule to run SUMMA simulations for each GRU chunk
rule run_summa_in_gru_chunks:
    wildcard_constraints:
        gru_chunk='|'.join(GRU_CHUNKS)
    input:
        file_manager = Path(config['file_manager'])
    output:
        summa_chunked_output = Path(config['summa_output_path'], f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc")
    params:
        summa_exe = config['summa_exe'],
        gru_start = lambda wildcards: ss_utils.extract_gru_int(wildcards.gru_chunk),
        gru_count = config['gru_chunk_size']
    log:
        log_file = Path(config['logging_path'], f"summa_{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}.log")
    run:
        sim = ps.Simulation(params.summa_exe, input.file_manager)
        sim_config = {
            'manager': {
                'simStartTime': config['sim_start_time'],
                'simEndTime': config['sim_end_time']
            }
        }
        sim.apply_config(sim_config)

        if params.gru_start + params.gru_count > num_grus:
            params.gru_count = num_grus - params.gru_start + 1

        sim.run(
            run_suffix=config['run_suffix'],
            startGRU=params.gru_start,
            countGRU=params.gru_count,
            write_config=False,
            write_manager=True,
            freq_restart='m'
        )

        print(f"SUMMA simulation for GRUs {params.gru_start} to {params.gru_start + params.gru_count - 1} finished with {sim.status}.")

# New rule to handle restart files per date and gru_chunk
"""
rule generate_restart_file:
    wildcard_constraints:
        gru_chunk='|'.join(GRU_CHUNKS),
        date_str='|'.join(DATE_STRINGS)
    input:
        # Depend on the completion of the SUMMA simulation for the gru_chunk
        summa_chunked_output = Path(config['summa_output_path'], f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc")
    output:
        restart_file = Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}_{{gru_chunk}}.nc")
    run:
        from pathlib import Path
        if not Path(output.restart_file).exists():
            Path(output.restart_file).touch()
            print(f"WARNING: {output.restart_file} not found.")

# Rule to merge restart files per date
rule merge_restart_files_per_date:
    wildcard_constraints:
        date_str='|'.join(DATE_STRINGS)
    input:
        chunked_restart_files = expand(
            Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}_{{gru_chunk}}.nc"),
            gru_chunk=GRU_CHUNKS,
            date_str="{date_str}"
        )
    output:
        merged_restart_file = Path(config['summa_output_path'], f"{config['case_name']}_restart_{{date_str}}_{config['run_suffix']}.nc")
    run:
        ss_utils.merge_netcdf_output(input.chunked_restart_files, output.merged_restart_file)
"""
# Rule to merge SUMMA output files
rule merge_summa_output_files:
    input:
        chunked_output_files = expand(
            Path(config['summa_output_path'], f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc"),
            gru_chunk=GRU_CHUNKS
        )
    output:
        merged_output_file = Path(config['summa_output_path'], f"{config['case_name']}_{config['run_suffix']}_timestep.nc")
    run:
        ss_utils.merge_netcdf_output(input.chunked_output_files, output.merged_output_file)

Overwriting ../rules/run_pysumma_base.smk


In [18]:
%%writefile ../rules/run_fsca_simulations.smk
"""

Snakemake file to calculate fSCA based on SUMMA model simulations.

"""

from pathlib import Path
import sys
# Import local packages
sys.path.append(str(Path('../').resolve()))
sys.path.append(str(Path('../../').resolve()))
sys.path.append(str(Path('../../snow_dist').resolve()))
from scripts import ss_utils
from snow_dist import summa_fsca_model

# Resolve all file paths and directories in the config file
config['base_dir'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa'
config['summa_output_dir'] = str(Path(config['base_dir'],'output'))
config['summa_interim_dir'] = str(Path(config['base_dir'],'interim'))
config['fsca_output_dir'] = str(Path(config['base_dir'],'fsca'))
config['plot_dir'] = str(Path(config['base_dir'],'plots'))

config_file_path = "/Users/drc858/GitHub/snow_dist/snakemake/config/fsca_snakemake_tuolumne_hydrofabric.yaml"
cv_list = [0.9]
fsca_models = ['ssnowd']

#Read first raw forcing file for easymore remapping
summa_day_result_filepaths = list(Path(config['summa_output_dir']).glob('*H51*.nc'))
summa_filenames = [Path(filepath).name for filepath in summa_day_result_filepaths]


def collect_netcdf_files(root_dir, filename_pattern):
    """Recursively find all netcdf files matching the filename pattern in the given root directory."""
    return list(Path(root_dir).rglob(filename_pattern))

# Prepare summa model results for fsca
# This includes summing the input and output snowpack variables
rule run_fsca_model:
    input:
        #Main file output
        expand(Path(config['fsca_output_dir'],"{fsca_model}","{cv}","{filenames}"), fsca_model=fsca_models, filenames=summa_filenames, cv=cv_list)
        #Plot file output

#Prepate fsca models from summa output
rule prepare_fsca_model_input:
    input:
        summa_output = Path(config['summa_output_dir'],"{filenames}")
    output:
        fsca_input = Path(config['summa_interim_dir'],"{filenames}")
    run:
        fsca_ds = summa_fsca_model.prepare_summa_fsca_input(input.summa_output, output.fsca_input)

# Run fsca settings
rule run_fsca_simulations:
    input:
        fsca_input = Path(config['summa_interim_dir'], "{filenames}")
    output:
        fsca_output = Path(config['fsca_output_dir'],"{fsca_model}","{cv}","{filenames}"),
    params:
        lumped_cv_value = lambda wildcards: wildcards.cv
    run:
        summa_fsca_model.calc_summa_fsca(input.fsca_input, params.lumped_cv_value, output.fsca_output)



Overwriting ../rules/run_fsca_simulations.smk


In [19]:
! snakemake -s ../rules/run_fsca_simulations.smk --configfile /Users/drc858/GitHub/snow_dist/snakemake/config/fsca_snakemake_tuolumne_hydrofabric.yaml --unlock
! snakemake -s ../rules/run_fsca_simulations.smk --cores 8 --configfile /Users/drc858/GitHub/snow_dist/snakemake/config/fsca_snakemake_tuolumne_hydrofabric.yaml

Unlocking working directory.
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 8
Rules claiming more threads will be scaled down.
Job stats:
job                         count
------------------------  -------
prepare_fsca_model_input       10
run_fsca_model                  1
run_fsca_simulations           11
total                          22

Select jobs to execute...

[Mon Apr 15 21:01:44 2024]
rule prepare_fsca_model_input:
    input: /Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_005_H51_timestep.nc
    output: /Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/interim/tuolumne_005_H51_timestep.nc
    jobid: 14
    reason: Missing output files: /Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/interim/tuolumne_005_H51_timestep.nc
    wildcards: filenames=tuolumne_005_H51_timestep.nc
    resources: tmpdir=/var/folders/cf/h3xm_pj57nxdnm6lk83rzxt9jt0gpj/T

[Mon Apr 15 21:01:44 2024]
rule prepare_fsca_model_input:
    i

In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt

def collect_netcdf_files(root_dir):
    """Recursively find all netcdf files in the given root directory."""
    netcdf_files = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(".nc"):
                netcdf_files.append(os.path.join(dirpath, filename))
    return netcdf_files

def plot_fsca_scalarSWE(netcdf_files, hru_value):
    """Load netcdf files, filter by hru and plot fsca and scalarSWE."""
    datasets = []
    for netcdf_file in netcdf_files:
        ds = xr.open_dataset(netcdf_file)
        datasets.append(ds)
    
    # Combine datasets into one
    combined_dataset = xr.concat(datasets, dim='time')
    
    # Filter by hru value
    hru_data = combined_dataset.sel(hru=hru_value)
    
    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    ax1.set_xlabel('Time')
    ax1.set_ylabel('fsca', color='tab:blue')
    ax1.plot(hru_data['time'], hru_data['fsca'], color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    
    ax2 = ax1.twinx()
    ax2.set_ylabel('scalarSWE', color='tab:red')
    ax2.plot(hru_data['time'], hru_data['scalarSWE'], color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    
    fig.tight_layout()
    plt.show()

# Sample usage:
# root_dir = "/path/to/directory"
# hru_value = some_value
# netcdf_files = collect_netcdf_files(root_dir)
# plot_fsca_scalarSWE(netcdf_files, hru_value)

# Note: The above code assumes that your NetCDF files have 'hru', 'fsca', 'scalarSWE', and 'time' dimensions/variables.
# Please modify accordingly if they have different names or structures.


In [313]:

import xarray as xr 
input_files = ['/Users/dcasson/Data/yukon_esp/output/Chena_restart_2001010200_base_G13-15.nc',
'/Users/dcasson/Data/yukon_esp/output/Chena_restart_2002010200_base_G16-18.nc',
'/Users/dcasson/Data/yukon_esp/output/Chena_restart_2002010200_base_G37-39.nc',
'/Users/dcasson/Data/yukon_esp/output/Chena_restart_2002010200_base_G28-30.nc']
output_file='/Users/dcasson/Data/yukon_esp/output/Chena_restart_2002010200.nc'

def merge_netcdf_output(input_file_list, output_file):
    """Merge a list of netcdf files into a single file"""
    out_ds = [xr.open_dataset(input_file) for input_file in input_file_list]
    hru_vars = [] # variables that have hru dimension
    gru_vars = [] # variables that have gru dimension
    for name, var in out_ds[0].variables.items():
        if 'hru' in var.dims:
            hru_vars.append(name)
        elif 'gru' in var.dims:
            gru_vars.append(name)
    hru_ds = [ds[hru_vars] for ds in out_ds]
    gru_ds = [ds[gru_vars] for ds in out_ds]
    hru_merged = xr.concat(hru_ds, dim='hru')
    gru_merged = xr.concat(gru_ds, dim='gru')

    merged_ds = xr.merge([hru_merged, gru_merged])
    merged_ds.to_netcdf(output_file)

    return merged_ds

merge_netcdf_output(input_files, output_file)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/dcasson/Data/yukon_esp/output/Chena_restart_2001010200_base_G13-15.nc'